In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Generate synthetic e-commerce sales data
def generate_ecommerce_data(n_records=2000):
    # Date range for the past 2 years
    start_date = datetime(2022, 1, 1)
    end_date = datetime(2024, 1, 1)
    date_range = pd.date_range(start_date, end_date, freq='D')
    
    # Product categories and their typical price ranges
    categories = {
        'Electronics': (50, 2000),
        'Clothing': (20, 300),
        'Books': (10, 80),
        'Home & Garden': (25, 500),
        'Sports': (30, 400),
        'Beauty': (15, 200),
        'Toys': (10, 150)
    }
    
    # Customer segments
    segments = ['Premium', 'Regular', 'Budget']
    segment_weights = [0.2, 0.5, 0.3]
    
    # Regions
    regions = ['North', 'South', 'East', 'West', 'Central']
    
    # Sales channels
    channels = ['Online', 'Retail Store', 'Mobile App']
    
    data = []
    
    for i in range(n_records):
        # Generate basic transaction info
        transaction_id = f'TXN_{str(i+1).zfill(6)}'
        date = np.random.choice(date_range)
        
        # Convert to Python datetime to access month attribute
        if hasattr(date, 'to_pydatetime'):
            date_obj = date.to_pydatetime()
        else:
            date_obj = pd.to_datetime(date).to_pydatetime()
        
        # Category affects price and seasonal trends
        category = np.random.choice(list(categories.keys()))
        price_range = categories[category]
        
        # Seasonal effects
        month = date_obj.month
        seasonal_multiplier = 1.0
        if category == 'Clothing':
            # Higher sales in spring/fall
            seasonal_multiplier = 1.3 if month in [3,4,9,10] else 0.8
        elif category == 'Electronics':
            # Higher sales in November/December (holiday season)
            seasonal_multiplier = 1.5 if month in [11,12] else 0.9
        elif category == 'Sports':
            # Higher sales in summer
            seasonal_multiplier = 1.4 if month in [5,6,7,8] else 0.8
        
        # Generate price with seasonal adjustment
        base_price = np.random.uniform(price_range[0], price_range[1])
        price = base_price * seasonal_multiplier
        
        # Quantity (higher prices tend to have lower quantities)
        if price > 500:
            quantity = np.random.randint(1, 3)
        elif price > 100:
            quantity = np.random.randint(1, 5)
        else:
            quantity = np.random.randint(1, 8)
        
        # Customer segment affects spending
        segment = np.random.choice(segments, p=segment_weights)
        if segment == 'Premium':
            price *= np.random.uniform(1.2, 1.8)
        elif segment == 'Budget':
            price *= np.random.uniform(0.6, 0.9)
        
        # Other attributes
        region = np.random.choice(regions)
        channel = np.random.choice(channels, p=[0.4, 0.35, 0.25])
        
        # Customer satisfaction (correlated with price and segment)
        base_satisfaction = np.random.normal(3.5, 0.8)
        if segment == 'Premium':
            base_satisfaction += 0.5
        elif segment == 'Budget':
            base_satisfaction -= 0.3
        
        # Higher priced items tend to have slightly higher satisfaction
        if price > 200:
            base_satisfaction += 0.2
        
        satisfaction = np.clip(base_satisfaction, 1, 5)
        
        # Discount percentage (varies by channel and season)
        if channel == 'Online':
            discount = np.random.uniform(0, 25)
        elif channel == 'Mobile App':
            discount = np.random.uniform(5, 30)  # Apps often have better deals
        else:
            discount = np.random.uniform(0, 15)
        
        # Holiday season discounts
        if month in [11, 12]:
            discount += np.random.uniform(5, 15)
        
        discount = min(discount, 50)  # Cap at 50%
        
        # Calculate final amounts
        subtotal = price * quantity
        discount_amount = subtotal * (discount / 100)
        total_amount = subtotal - discount_amount
        
        # Return customer (higher satisfaction tends to create return customers)
        return_customer = np.random.choice([True, False], 
                                         p=[0.7 if satisfaction > 4 else 0.3, 
                                           0.3 if satisfaction > 4 else 0.7])
        
        data.append({
            'transaction_id': transaction_id,
            'date': date,
            'category': category,
            'price': round(price, 2),
            'quantity': quantity,
            'subtotal': round(subtotal, 2),
            'discount_percent': round(discount, 1),
            'discount_amount': round(discount_amount, 2),
            'total_amount': round(total_amount, 2),
            'customer_segment': segment,
            'region': region,
            'sales_channel': channel,
            'customer_satisfaction': round(satisfaction, 1),
            'return_customer': return_customer,
            'month': month,
            'quarter': f'Q{(month-1)//3 + 1}',
            'day_of_week': date_obj.strftime('%A'),
            'is_weekend': date_obj.weekday() >= 5
        })
    
    return pd.DataFrame(data)

# Generate the dataset
df = generate_ecommerce_data(2000)

# Save to CSV
df.to_csv('ecommerce_sales_data.csv', index=False)

print("Dataset generated successfully!")
print(f"Shape: {df.shape}")
print("\nFirst few rows:")
print(df.head())
print("\nDataset info:")
print(df.info())
print("\nNumerical columns summary:")
print(df.describe())

Dataset generated successfully!
Shape: (2000, 18)

First few rows:
  transaction_id       date       category   price  quantity  subtotal  \
0     TXN_000001 2022-04-13  Home & Garden  371.29         3   1113.87   
1     TXN_000002 2023-05-07         Beauty   75.05         4    300.19   
2     TXN_000003 2023-11-13  Home & Garden  487.53         3   1462.60   
3     TXN_000004 2022-10-01  Home & Garden  326.71         2    653.42   
4     TXN_000005 2023-03-04         Sports  183.39         3    550.16   

   discount_percent  discount_amount  total_amount customer_segment   region  \
0               0.5             5.73       1108.14           Budget    South   
1               4.4            13.11        287.08          Premium  Central   
2              33.3           487.07        975.53          Regular     East   
3               6.0            39.37        614.05           Budget  Central   
4               4.6            25.42        524.73           Budget     West   

  sales

In [3]:
df.head()

,transaction_id,date,category,price,quantity,subtotal,discount_percent,discount_amount,total_amount,customer_segment,region,sales_channel,customer_satisfaction,return_customer,month,quarter,day_of_week,is_weekend
0,TXN_000001,2022-04-13,Home & Garden,371.29,3,1113.87,0.5,5.73,1108.14,Budget,South,Online,4.7,False,4,Q2,Wednesday,False
1,TXN_000002,2023-05-07,Beauty,75.05,4,300.19,4.4,13.11,287.08,Premium,Central,Retail Store,4.6,True,5,Q2,Sunday,True
2,TXN_000003,2023-11-13,Home & Garden,487.53,3,1462.60,33.3,487.07,975.53,Regular,East,Online,3.7,False,11,Q4,Monday,False
3,TXN_000004,2022-10-01,Home & Garden,326.71,2,653.42,6.0,39.37,614.05,Budget,Central,Online,3.7,False,10,Q4,Saturday,True
4,TXN_000005,2023-03-04,Sports,183.39,3,550.16,4.6,25.42,524.73,Budget,West,Online,5.0,False,3,Q1,Saturday,True
